# Notebook 04 — Protein Embedding (ESM-2)

Pre-computes ESM-2 embeddings for the 6 target proteins. Run **once** — output is a fixed tensor reused by notebook 05.

| Step | Description |
|------|-------------|
| 1 | Fetch canonical sequences from UniProt REST API |
| 2 | Load ESM-2 (frozen) and extract per-residue embeddings |
| 3 | Mean-pool residue tokens → 1280-dim vector per protein |
| 4 | Sanity check: cosine similarity between proteins |
| 5 | Save as `all_proteins_esm2_mean.pt` |

> **ESM-2 is kept frozen** throughout — only 6 sequences, purpose is to inject biological context, not fine-tune a protein LM.  
> GPU recommended but not required (6 short sequences, runs in <2 min on CPU).

---
## 0 — Imports & Config

In [ ]:
import sys, time, requests
import numpy as np
import torch
import yaml
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from transformers import AutoTokenizer, AutoModel

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

with open(REPO_ROOT / 'config.yaml') as f:
    CFG = yaml.safe_load(f)

MOL_REP_ROOT = Path(CFG['paths']['mol_rep_root'])
OUT_DIR = MOL_REP_ROOT / 'protein_embeddings'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ESM-2 model variants (choose one):
# 'facebook/esm2_t6_8M_UR50D'    →  320-dim  (fast, demo)
# 'facebook/esm2_t12_35M_UR50D'  →  480-dim
# 'facebook/esm2_t30_150M_UR50D' →  640-dim
# 'facebook/esm2_t33_650M_UR50D' → 1280-dim  ← used in paper
ESM2_MODEL = 'facebook/esm2_t33_650M_UR50D'

print(f'Device    : {DEVICE}')
print(f'ESM2 model: {ESM2_MODEL}')
print(f'Output    : {OUT_DIR}')

---
## 1 — Protein Sequences

Sequences are fetched automatically from UniProt. If offline, the hardcoded fallback sequences below are used.

In [ ]:
# UniProt IDs — canonical human sequences
PROTEINS = {
    'AR_IC50':         {'uniprot_id': 'P10275', 'name': 'Androgen receptor'},
    'CYP19A1_IC50':    {'uniprot_id': 'P11511', 'name': 'Aromatase (CYP19A1)'},
    'ESR1_alpha_IC50': {'uniprot_id': 'P03372', 'name': 'Estrogen receptor alpha'},
    'ESR2_beta_IC50':  {'uniprot_id': 'Q92731', 'name': 'Estrogen receptor beta'},
    'HSD11B1_IC50':    {'uniprot_id': 'P28845', 'name': '11-beta-HSD1'},
    'PPARG_IC50':      {'uniprot_id': 'P37231', 'name': 'PPARgamma'},
}

# Hardcoded fallback (first 50 AA shown — full sequences truncated for readability)
# Replace with full canonical sequences if UniProt fetch fails.
FALLBACK_SEQUENCES = {
    'AR_IC50':         None,  # set manually if needed
    'CYP19A1_IC50':    None,
    'ESR1_alpha_IC50': None,
    'ESR2_beta_IC50':  None,
    'HSD11B1_IC50':    None,
    'PPARG_IC50':      None,
}


def fetch_sequence(uniprot_id: str, retries: int = 3) -> str:
    """Fetch canonical FASTA sequence from UniProt REST API."""
    url = f'https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta'
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            lines = r.text.strip().split('\n')
            return ''.join(lines[1:])  # skip FASTA header
        except Exception as e:
            if attempt < retries - 1:
                time.sleep(2)
            else:
                raise RuntimeError(f'UniProt fetch failed for {uniprot_id}: {e}')


print('Fetching sequences from UniProt...')
for task, info in PROTEINS.items():
    uid = info['uniprot_id']
    try:
        seq = fetch_sequence(uid)
        info['sequence'] = seq
        print(f'  ✅ {task:<22} | {uid} | {len(seq)} aa')
    except Exception as e:
        fallback = FALLBACK_SEQUENCES.get(task)
        if fallback:
            info['sequence'] = fallback
            print(f'  ⚠️  {task:<22} | {uid} | fetch failed, using fallback ({len(fallback)} aa)')
        else:
            info['sequence'] = None
            print(f'  ❌ {task:<22} | {uid} | FAILED: {e}')

---
## 2 — Load ESM-2 (Frozen)

In [ ]:
print(f'Loading {ESM2_MODEL} ...')
tokenizer = AutoTokenizer.from_pretrained(ESM2_MODEL)
esm2      = AutoModel.from_pretrained(ESM2_MODEL).to(DEVICE)

# Freeze all parameters — ESM-2 is used as a fixed feature extractor
for param in esm2.parameters():
    param.requires_grad = False
esm2.eval()

n_params = sum(p.numel() for p in esm2.parameters()) / 1e6
print(f'✅ ESM-2 loaded | {n_params:.0f}M params | frozen=True | device={DEVICE}')

---
## 3 — Extract Embeddings (Mean-Pool over Residues)

Exclude `[CLS]` (index 0) and `[EOS]` (last token) — mean-pool over residue tokens only.

In [ ]:
def extract_mean_embedding(sequence: str) -> torch.Tensor:
    """Returns mean-pooled residue embedding: shape (d_esm,)."""
    inputs = tokenizer(
        sequence, return_tensors='pt',
        add_special_tokens=True
    ).to(DEVICE)

    with torch.no_grad():
        out = esm2(**inputs)

    # last_hidden_state: (1, L+2, d)  — index 0=[CLS], -1=[EOS]
    residue_emb = out.last_hidden_state[0, 1:-1, :]  # (L, d)
    return residue_emb.mean(dim=0).cpu().float()      # (d,)


embeddings = {}   # task_name -> (d,) tensor

print(f'Extracting embeddings...')
print(f'{"Task":<22} {"UniProt":>8} {"Length":>8} {"Emb dim":>9} {"Time":>7}')
print('-' * 60)

for task, info in PROTEINS.items():
    seq = info.get('sequence')
    if seq is None:
        print(f'  ❌ {task} — skipped (no sequence)')
        continue

    t0  = time.time()
    emb = extract_mean_embedding(seq)
    embeddings[task] = emb
    print(f'  {task:<22} {info["uniprot_id"]:>8} {len(seq):>8} {emb.shape[0]:>9} {time.time()-t0:>6.2f}s')

print(f'\n✅ Done: {len(embeddings)}/6 proteins embedded.')

---
## 4 — Sanity Check: Pairwise Cosine Similarity

Expected: ESR1 ↔ ESR2 high similarity (same receptor family), AR lower, HSD11B1 & CYP19A1 distinct.

In [ ]:
import torch.nn.functional as F

task_names  = list(embeddings.keys())
short_names = [t.replace('_IC50', '').replace('_alpha', 'α').replace('_beta', 'β') for t in task_names]
emb_matrix  = torch.stack([embeddings[t] for t in task_names])  # (6, d)

# Normalise → cosine similarity matrix
emb_norm = F.normalize(emb_matrix, dim=1)
cos_sim  = (emb_norm @ emb_norm.T).numpy()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cos_sim,
    xticklabels=short_names, yticklabels=short_names,
    annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1,
    ax=ax, linewidths=0.5, linecolor='white'
)
ax.set_title('Pairwise Cosine Similarity — ESM-2 Mean Embeddings', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'protein_cosine_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCosine similarity matrix:')
import pandas as pd
print(pd.DataFrame(cos_sim, index=short_names, columns=short_names).round(3).to_string())

---
## 5 — Save

Saved as a dict `{task_name: tensor(d,)}` — loaded directly in notebook 05 as a fixed lookup.

In [ ]:
out_path = OUT_DIR / 'all_proteins_esm2_mean.pt'
torch.save(embeddings, out_path)
print(f'✅ Saved → {out_path}')

# Verify reload
reloaded = torch.load(out_path)
print('\nReload check:')
for task, emb in reloaded.items():
    print(f'  {task:<22} shape={tuple(emb.shape)} | mean={emb.mean():.4f} | std={emb.std():.4f}')

---
## ✅ Notebook 04 Complete

**Output:**
```
{mol_rep_root}/protein_embeddings/
├── all_proteins_esm2_mean.pt        ← main output (used by nb05)
└── protein_cosine_similarity.png    ← sanity check figure
```

➡️ **Next:** [05_model_training.ipynb](05_model_training.ipynb)